# Task 1: Artistic Style Transfer in Colorization
### Internship Task - Real-time Gen AI Image Colorization

Artistic Style Transfer in Colorization Description: In this job, you will create a model that colorizes grayscale photos while simultaneously applying a specific artistic style to the colorized output. Guidelines: This exercise examines your ability to mix image colorization and style transfer approaches. The model should let users select from predetermined artistic styles. A graphical user interface (GUI) is required to allow users to submit photographs and choose styles.

In [ ]:
!pip install -r "d:\Downloads\image_gen_colorization\requirements.txt"



In [ ]:
# Imports
import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

import torch
import numpy as np
from PIL import Image
import cv2
import os
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from diffusers.utils import load_image
import gradio as gr

# Device Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Using device: {device} with {dtype}")

A matching Triton is not available, some optimizations will not be enabled.
Error caught was: No module named 'triton'


Using device: cuda with torch.float16


## 1. Load Models
We load the **Canny ControlNet** model and the **Stable Diffusion v1.5** base model.

In [ ]:
# Load ControlNet
controlnet = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_canny", torch_dtype=dtype)

# Load Pipeline
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", controlnet=controlnet, torch_dtype=dtype
)

# Set Scheduler
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to(device)

# Memory Optimizations
if device == "cuda":
    pipe.enable_model_cpu_offload()
    pipe.enable_xformers_memory_efficient_attention()
    
print("Models loaded successfully!")

d:\Downloads\image_gen_colorization\.venv\lib\site-packages\huggingface_hub\file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
text_encoder\model.safetensors not found


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["id2label"]` will be overriden.


Models loaded successfully!


## 2. Preprocessing
We need to convert our grayscale image into a **Canny edge map** which ControlNet uses as a guide.

In [ ]:
def get_canny_image(image, low_threshold=100, high_threshold=200):
    # Convert to numpy
    image = np.array(image)
    # Extract edges
    image = cv2.Canny(image, low_threshold, high_threshold)
    image = image[:, :, None]
    image = np.concatenate([image, image, image], axis=2)
    return Image.fromarray(image)

## 3. Artistic Stylization Function
This function applies color and style to the grayscale image.

In [ ]:
def stylize_image(image, style, custom_prompt=""):
    # Preprocess
    canny_image = get_canny_image(image)
    
    # Define Style Prompts
    style_dict = {
        "Van Gogh": "in the style of Vincent van Gogh, oil painting, expressive brushstrokes, vibrant colors",
        "Watercolor": "soft watercolor painting, bleeding colors, artistic, paper texture",
        "Cyberpunk": "cyberpunk style, neon lights, futuristic colors, digital art",
        "Sketch to Color": "pencil sketch colored with markers, artistic strokes, vibrant"
    }
    
    style_prompt = style_dict.get(style, "highly detailed colorized photo")
    full_prompt = f"{custom_prompt}, {style_prompt}, masterpiece, cinematic lighting"
    negative_prompt = "monochrome, grayscale, b&w, low quality, bad anatomy, blurry"
    
    # Generate
    output = pipe(
        full_prompt, 
        image=canny_image, 
        negative_prompt=negative_prompt, 
        num_inference_steps=20,
        controlnet_conditioning_scale=1.0
    ).images[0]
    
    return output

## 4. Gradio GUI
We create a simple interface to demonstrate the feature.

In [ ]:
with gr.Blocks(title="Task 1: Artistic Style Transfer") as demo:
    gr.Markdown("# 🎨 Artistic Style Transfer in Colorization")
    gr.Markdown("Upload a grayscale photo and choose an artistic style to apply during colorization.")
    
    with gr.Row():
        with gr.Column():
            input_img = gr.Image(label="Upload Grayscale Image", type="pil")
            style_opt = gr.Dropdown(
                choices=["Van Gogh", "Watercolor", "Cyberpunk", "Sketch to Color", "Natural Realism"], 
                value="Van Gogh",
                label="Choose Style"
            )
            prompt_inp = gr.Textbox(label="Subject Description", placeholder="e.g., a landscape with mountains")
            btn = gr.Button("Colorize with Style", variant="primary")
        
        with gr.Column():
            output_img = gr.Image(label="Artistic Colored Output")
            
    btn.click(fn=stylize_image, inputs=[input_img, style_opt, prompt_inp], outputs=output_img)

demo.launch(share=True, debug=False)

Running on local URL:  http://127.0.0.1:7861
IMPORTANT: You are using gradio version 3.50.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://66f5b581d1d3d9989a.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
